### RTDL Baselines

This notebook trains the RTDL MLP, ResNet, and FT-Transformer benchmark models for SMT comparison on the shared transformed stop dataset using the shared inverse-transform evaluation path. The trained models and their training histories are saved to the shared models directory, and the evaluation metrics are saved to the shared benchmarks directory.

In [ ]:
# %reset -f
%load_ext autoreload
%autoreload complete --log


In [ ]:
from smtgraphformer import *
from smtgraphformer.benchmarks.rtdl import *
from smtgraphformer.modelAdapters import createCanonicalBuilds

setDisplayOptions()
sr = setReproducibility(17711)


In [ ]:
root = Path("../data")
fp = root.joinpath("atbData-May2024-stopLevel-[fPM.eST.eLU.eDW].pkl")
assert fp.exists(), "!!!"


### Shared Artefacts

In [ ]:
builder = createCanonicalBuilds(fp)
# ---
dataBCS = builder.canonicalStops
ds_splits = builder.splitPlan
bundle = builder.transformBundle
dataFTB = builder.transformedStops


In [ ]:
sz_samples = {"train": 2048, "valid": 512, "test": 512}
l_samples = [
    dsub.sample(sz_samples[str(s)], random_state=17711)
    for s, dsub in dataFTB.groupby("$split", observed=True)
]
dataSmoke = pd.concat(l_samples).sort_index()


In [ ]:
# rtdlData = tfmStopLevelRTDL(dataSmoke, bundle)
rtdlData = tfmStopLevelRTDL(dataFTB, bundle)
printFeatureCardinality(rtdlData)


### Training and Evaluation

In [ ]:
configs = [
    ("MLP", MLPConfig(max_epochs=100, patience=20)),
    ("RNet", RNetConfig(max_epochs=100, patience=20)),
    ("FTT", FTTConfig(max_epochs=100, patience=20)),
]


In [ ]:
results = {}
l_metrics = []

for name, cfg in configs:
    print(f"running {name} ...")
    with Timer(verbose=False):
        i_model, i_metrics, i_comparisons = runBaselineRTDL(dataFTB, bundle, cfg, raw_evaluation=True)
        i_model.save()
        df_csver(i_model.history, tag=f"{i_model.cfg.model_dir}/history{name}")

    results[name] = {"model": i_model, "metrics": i_metrics, "comparisons": i_comparisons}
    l_metrics.append(i_metrics)


In [ ]:
metrics = pd.concat(l_metrics, ignore_index=True)
df_csver(metrics, "../models/benchmarks/metricsRTDL")
print(metrics.tail(4))


In [ ]:
# display(results["FTT"]["comparisons"]["test"].head().round().astype(int))


### end